# 03b - Similarity and Nearest Neighbors

## Last time: distance depends on representation

In Lecture 03a, we represented each utility as a row of selected numerical features. We used a **distance metric** to compare those rows, and smaller distances meant more similar utilities.

The nearest utility changed when we changed the feature scales or the distance metric. That was the central lesson: a numerical distance is created by our **representation** (the numerical features chosen to describe each utility), scaling, and comparison choices. It is not a natural fact waiting inside the data.

We also learned to exclude an observation's distance to itself, inspect a near tie, and test whether a result survives a reasonable alternative. Today we use those ideas to move from one nearest peer to a neighborhood.

## From one nearest record to a neighborhood

Lecture 03a stopped after finding one nearest peer. Keeping several nearby observations gives us a **neighborhood**: a small local group selected around one observation.

A ranking tells us who comes first, second, and third. A neighborhood changes the result we inspect. Instead of treating one winner as the answer, we retain several nearby observations so we can compare what they have in common, where they differ, and whether the last included observation is meaningfully closer than the next one. We still use a ranking to build the neighborhood, but the neighborhood gives us local context that one rank cannot.

That extension creates a new problem. A ranking will always produce a first, second, and third place, even when the observations are not meaningfully close. Different defensible meanings of similarity can also produce different neighborhoods.

KNN regression and classification may be familiar. Those methods begin by constructing a neighborhood, then use the neighbors' known values or labels to predict an unknown outcome. We do not need that predictive machinery here. We use the neighborhood itself for analytical inference: reasoning from comparable observations about local structure. This does not replace supervised KNN; it examines the foundation that supervised KNN builds on.

Our general question is:

> What does a nearest-neighbor result actually tell us, and how does our chosen meaning of similarity shape it?

## Building a Neighborhood

Start with one observation we want to understand. Decide which other observations belong in the comparison, and describe every observation with the same features in the same order. Then compare the observation of interest with each of the others.

Each comparison produces a numerical score. A **distance** or **dissimilarity** gets smaller as two observations become more alike; a **similarity** gets larger. Sort the scores in the appropriate direction, leave out the observation's comparison with itself, and retain the first few observations as its neighborhood.

A score tells us about one pair. A rank tells us where that pair falls relative to the other pairs. Keeping the top few turns the ranking into a neighborhood. Being returned near the top does not, by itself, prove that an observation is close enough or useful for our purpose.

## Example Dataset

Let's start with four customer profiles. Each row describes one customer's quantities across four products. Every value is visible, so we can predict the result, calculate it, and explain why it changes.

The first row is the customer we want to understand, so we call it the customer of interest. The other names describe what makes each comparison useful: one customer has the same proportions, one concentrates on the dominant product, and one buys the same set of products. The four product columns stay in the same order in each table and chart so the quantities line up.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from scipy.spatial.distance import cdist

PRODUCTS = ["Product 1", "Product 2", "Product 3", "Product 4"]
PRODUCT_COLORS = ["#4477AA", "#EE6677", "#228833", "#CCBB44"]

example_profiles = pd.DataFrame(
    {
        "Product 1": [10, 100, 10, 1],
        "Product 2": [1, 10, 0, 1],
        "Product 3": [1, 10, 0, 1],
        "Product 4": [1, 10, 0, 1],
    },
    index=[
        "Customer of interest",
        "Same proportions",
        "Dominant product",
        "Same product set",
    ],
)
example_profiles

The DataFrame gives every customer the same four product columns. The row names state the role each profile will play in the comparison.

### Quantity Profiles

We want to compare both the overall quantity purchased and how each customer distributes that quantity across products. The figure places the same products in the same vertical positions for every customer and uses one horizontal scale across all four panels. The table preserves the exact values, while the bars make differences in volume and concentration easier to see.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6), sharex=True, sharey=True)

for axis, (profile, values) in zip(axes, example_profiles.iterrows(), strict=True):
    axis.barh(PRODUCTS, values, color=PRODUCT_COLORS, edgecolor="black", linewidth=0.5)
    axis.set_title(profile)
    axis.set_xlabel("Quantity")
    axis.grid(axis="x", alpha=0.2)

fig.suptitle("The same four product positions in each customer profile", y=1.02)
fig.tight_layout()
plt.show()

### Pause: predict two notions of resemblance

Which customer resembles how the customer of interest allocates quantities across products? Which customers bought exactly the same set of products? We call that set a customer's **product repertoire**. Explain why the two questions need not produce the same ordering.

##### Answer

The same-proportions customer preserves the allocation exactly at ten times the volume and buys the complete product repertoire. The dominant-product customer concentrates on the largest purchase and shares only one product. The same-product-set customer buys the complete repertoire but allocates quantity evenly. Allocation and repertoire therefore produce different orderings.

## Cosine Similarity

Cosine similarity compares the direction of two vectors. A vector is an ordered list of numbers, so we can picture a two-number vector as an arrow on a two-dimensional coordinate plane.

Start with $x=[2,1]$ and $y=[1,2]$. Their **dot product** multiplies values in matching positions and adds the products:

$$
x^\mathsf{T}y=(2\times1)+(1\times2)=4.
$$

The dot product measures how much one vector acts in the direction of the other.

The dot product becomes larger when the vectors point in similar directions, but it also grows when either vector becomes longer. The **length**, or Euclidean norm, of each vector is

$$
\lVert x\rVert_2=\lVert y\rVert_2=\sqrt{2^2+1^2}=\sqrt{5}.
$$

Cosine similarity divides the dot product by both lengths:

$$
s_{\cos}(x,y)=\frac{x^\mathsf{T}y}{\lVert x\rVert_2\lVert y\rVert_2}=\frac{4}{\sqrt{5}\sqrt{5}}=0.8.
$$

The value is the cosine of the angle between the arrows. Vectors pointing in the same direction have cosine similarity 1. Perpendicular vectors have similarity 0, and vectors pointing in opposite directions have similarity $-1$.

The following code creates a figure to illustrate this.

In [ ]:
simple_x = np.array([2, 1])
simple_y = np.array([1, 2])

# For one-dimensional NumPy arrays, @ calculates the dot product.
simple_dot = simple_x @ simple_y

# Calculate each vector's Euclidean length.
simple_x_length = np.linalg.norm(simple_x)
simple_y_length = np.linalg.norm(simple_y)
simple_cosine = simple_dot / (simple_x_length * simple_y_length)

# Convert the cosine similarity to an angle in degrees for the figure.
simple_angle = np.degrees(np.arccos(simple_cosine))

fig, axis = plt.subplots(figsize=(8.5, 4.2))

for vector, label, color in [
    (simple_x, r"$x=[2,1]$", "#4477AA"),
    (simple_y, r"$y=[1,2]$", "#CC6677"),
]:
    axis.quiver(
        0,
        0,
        vector[0],
        vector[1],
        angles="xy",
        scale_units="xy",
        scale=1,
        color=color,
        width=0.012,
        label=label,
    )

angle_x = np.arctan2(simple_x[1], simple_x[0])
angle_y = np.arctan2(simple_y[1], simple_y[0])
arc_angles = np.linspace(angle_x, angle_y, 100)
axis.plot(0.7 * np.cos(arc_angles), 0.7 * np.sin(arc_angles), color="#555555")
axis.text(0.62, 0.62, f"{simple_angle:.1f}°", ha="center")
axis.text(
    2.7,
    1.65,
    f"dot product = {simple_dot}\ncosine similarity = {simple_cosine:.1f}",
    fontsize=11,
)
axis.set(
    xlim=(-0.2, 5.8),
    ylim=(-0.2, 2.8),
    xlabel="First coordinate",
    ylabel="Second coordinate",
    title="Cosine similarity measures the angle between two vectors",
)
axis.set_aspect("equal")
axis.grid(alpha=0.2)
axis.legend(loc="upper right", frameon=False)
plt.show()

The `@` operator calculates the dot product by multiplying aligned coordinates and adding them. `np.linalg.norm` calculates each vector's length. The figure draws both vectors from the same origin and marks their 36.9-degree angle.

### Applied to Our Example

Each customer profile is a vector with one coordinate for each product. Its direction describes the relative pattern of quantities across those coordinates. Multiplying every quantity by the same positive number makes the vector longer without changing that pattern or direction.

We want to make those relative patterns easy to compare visually. We express each product quantity as a share of the customer's total quantity, then draw the shares as a 100% stacked bar. This gives every bar the same overall width while preserving the direction of its vector. Each colored segment shows one product's share of the total.

In [ ]:
composition = example_profiles.div(example_profiles.sum(axis=1), axis=0)
axis = composition.plot.barh(
    stacked=True,
    figsize=(10, 3.8),
    color=PRODUCT_COLORS,
    edgecolor="white",
)
axis.set_xlabel("Share of the profile's total quantity")
axis.set_ylabel("")
axis.set_title("Product allocation after removing total-volume differences")
axis.legend(title="Product", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

`example_profiles.sum(axis=1)` calculates each customer's total quantity. `.div(..., axis=0)` divides every product quantity by the total for its row, so each bar shows how one customer allocated purchases across the four products.

### Pause: interpret the proportional profiles

Which two customers have identical distributions across the four products? Based on this figure, what cosine similarity would you expect between them, and why?

##### Answer

The customer of interest and the same-proportions customer have identical distributions. One profile is ten times the other, so their vectors point in the same direction. We therefore expect their cosine similarity to equal 1.

### Cosine Calculations

The chart and cosine remove scale in different ways: the chart divides each row by its total quantity, while cosine divides the dot product by both vectors' Euclidean lengths. Both preserve the direction of a vector when every quantity is multiplied by the same positive number.

In the cosine formula, the numerator rewards quantity aligned in the same product positions. The denominator removes the effect of multiplying an entire profile by a positive constant.

In [ ]:
interest_vector = example_profiles.loc["Customer of interest"].to_numpy()
cosine_rows = []

for other_customer in example_profiles.index.drop("Customer of interest"):
    other_vector = example_profiles.loc[other_customer].to_numpy()
    dot_product = interest_vector @ other_vector
    interest_length = np.linalg.norm(interest_vector)
    other_length = np.linalg.norm(other_vector)
    cosine_rows.append(
        {
            "Customer": other_customer,
            "Dot product": dot_product,
            "Interest vector length": interest_length,
            "Other vector length": other_length,
            "Cosine similarity": dot_product / (interest_length * other_length),
        }
    )

cosine_example = pd.DataFrame(cosine_rows).set_index("Customer")
cosine_example.round(3)

The loop compares the customer of interest with each other row. For each pair, `@` calculates the dot product and `np.linalg.norm` calculates both lengths. The resulting DataFrame keeps those intermediate values beside the cosine similarity so the calculation remains inspectable.

The same-proportions customer has cosine similarity 1 because its vector points in exactly the same direction. Cosine also ranks the dominant-product customer above the same-product-set customer, approximately 0.985 to 0.640, because the largest quantities occupy the same product position.

The data did not choose cosine. We did. Cosine answers a question in which relative purchasing mix matters and total volume does not.

## Jaccard Similarity

Jaccard similarity provides an alternative interpretation of difference and similarity.

Jaccard similarity compares the overlap between two sets. For sets $X$ and $Y$, it divides the size of their intersection by the size of their union:

$$
J(X,Y)=\frac{|X\cap Y|}{|X\cup Y|}.
$$

Suppose $X=\{\text{Product 1},\text{Product 2},\text{Product 3}\}$ and $Y=\{\text{Product 2},\text{Product 3},\text{Product 4}\}$. Their intersection contains two products (Products 2 and 3), while their union contains all four. Their Jaccard similarity is therefore $2/4=0.5$.

Jaccard similarity equals 1 when the sets contain exactly the same elements and 0 when they contain no elements in common. Elements absent from both sets do not enter the union, so shared absence does not increase the similarity.

### Applied to Our Example

Now change the question. Convert each positive quantity to purchased or not purchased. Repeated purchases and total volume disappear; product presence remains.

The following code creates a figure to illustrate this.

In [ ]:
example_presence = example_profiles.gt(0).astype(int)

fig, axis = plt.subplots(figsize=(8, 3.3))
axis.imshow(example_presence, cmap=ListedColormap(["#F2F2F2", "#4477AA"]), aspect="auto")
axis.set_xticks(range(len(PRODUCTS)), PRODUCTS)
axis.set_yticks(range(len(example_presence)), example_presence.index)
axis.set_title("Product presence in the four customer profiles")

for row in range(example_presence.shape[0]):
    for column in range(example_presence.shape[1]):
        present = example_presence.iat[row, column]
        axis.text(
            column,
            row,
            "purchased" if present else "not purchased",
            ha="center",
            va="center",
            color="white" if present else "#333333",
            fontsize=8,
        )

fig.tight_layout()
plt.show()

`.gt(0)` tests whether each product quantity is positive. `.astype(int)` displays the resulting `True` and `False` values as 1 and 0 in the table used by the figure. Quantity no longer affects the comparison once a purchase is marked as present.

### Pause: interpret product presence

Which customers bought exactly the same set of products as the customer of interest? Which customer shares only one of its four products? Based on product presence, what Jaccard ordering do you expect?

##### Answer

The same-proportions and same-product-set customers share the complete four-product repertoire, so we expect both to have Jaccard similarity 1. The dominant-product customer shares only Product 1, so we expect it to rank last with similarity $1/4$.

### Jaccard Calculations

For these purchased-product sets, the intersection contains products bought by both customers and the union contains products bought by either customer. Products bought by neither customer do not enter the union.

In [ ]:
interest_set = set(
    np.array(PRODUCTS)[example_presence.loc["Customer of interest"].eq(1)]
)
jaccard_rows = []

for other_customer in example_presence.index.drop("Customer of interest"):
    other_set = set(np.array(PRODUCTS)[example_presence.loc[other_customer].eq(1)])

    # & finds the intersection; | finds the union.
    intersection = interest_set & other_set
    union = interest_set | other_set
    jaccard_rows.append(
        {
            "Customer": other_customer,
            "Shared products": len(intersection),
            "Products in either profile": len(union),
            "Jaccard similarity": len(intersection) / len(union),
        }
    )

jaccard_example = pd.DataFrame(jaccard_rows).set_index("Customer")
jaccard_example.round(3)

The `&` operator finds products in both sets, while `|` finds products in either set. The loop counts those sets for each comparison and divides the intersection count by the union count.

Jaccard gives both the same-proportions and same-product-set customers similarity 1. The dominant-product customer shares one of four products with the customer of interest, for 0.25. This reverses the dominant-product and same-product-set ordering produced by cosine. The customers did not change; the analytical question did.

## Two Meanings of Similarity

Cosine and Jaccard can compare the same customers, but they answer different questions because they preserve different information.

| Method | Question it answers | What matters | What does not matter |
| --- | --- | --- | --- |
| Cosine similarity | Are quantities distributed across products in similar proportions? | The direction of the quantity vectors | Multiplying every quantity in a profile by the same positive number |
| Jaccard similarity | Did the customers buy many of the same products? | Shared purchases relative to all products bought by either customer | Purchase quantities and products bought by neither customer |

In our example, cosine places the same-proportions customer first and ranks the dominant-product customer above the same-product-set customer. Jaccard ties the same-proportions and same-product-set customers, then ranks the dominant-product customer last.

The results do not conflict. Each method defines similarity differently. Before choosing one, decide whether resemblance should mean a similar allocation of quantities or a similar set of observed items.

In a later lecture, we will cover a related topic called association analysis. Both Jaccard similarity and association analysis can work with product-presence data, but they use it differently. Here, Jaccard compares customer rows to find customers who bought many of the same products. Association analysis looks across many rows for patterns among the product columns, such as products that tend to occur together. One produces similarities between observations; the other produces itemsets or rules. We will return to that distinction when we study association analysis.

A customer neighborhood can also become an input to a recommendation system, but that system answers another question. Here, we rank customers by resemblance and stop. A recommendation system would continue by ranking products that might be relevant to a customer.

## The Retail Case

The [UCI Online Retail dataset](https://doi.org/10.24432/C5BW33) records transactions for a UK-based non-store retailer and is distributed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). For this analysis, we use December 2010 purchases by identified United Kingdom customers. Cancellations and nonpositive quantities have been removed, and each row records one customer's total quantity for one product.

The source and preparation choices are documented in [`data/README.md`](data/README.md).

Suppose a manager asks, “We are reviewing customer 15032. Find three other customers who bought most like this customer during December, and explain what makes them similar.”

The phrase *bought most like* needs an analytical definition. It could mean that customers allocated their quantities across products in similar proportions, or that they bought many of the same products. We will represent the purchases in both ways, use cosine and Jaccard similarity to construct two neighborhoods, and then inspect the results to explain what each method found.

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/olearydj/INSY7130/"
    "main/lectures/03b-similarity-and-nearest-neighbors/"
    "data/03b-online-retail-december-customer-products.csv"
)
CUSTOMER_OF_INTEREST = 15032
K = 3

retail = pd.read_csv(DATA_URL, dtype={"stock_code": "string"})

dataset_size = pd.Series(
    {
        "Customer-product pairs": len(retail),
        "Customers": retail["customer_id"].nunique(),
        "Products": retail["stock_code"].nunique(),
    },
    name="Value",
)
dataset_size.to_frame()

`len(retail)` counts the customer-product pairs. `.nunique()` counts the distinct customer and product identifiers represented by those rows.

The 21,280 rows describe 815 customer profiles across 2,375 possible product positions.

## From four products to 2,375

The manager has identified customer `15032` for review. The other 814 December customers form the **reference set**, the observations available for comparison. A pivot places one customer on each row and one product on each column. A zero then means that the customer did not buy that product in the retained December records.

This customer-by-product matrix is a **representation** of the purchase records: a particular way of describing each customer for comparison. We create two aligned representations from it. The first keeps product quantities. The second converts every positive quantity to `True`, meaning purchased, and every zero to `False`, meaning not purchased.

The following table shows the structure before we work with the complete matrix. It contains customer `15032`, the first three customer IDs that share at least one product with that customer, and eight products bought by at least two of the four customers.

In [ ]:
quantities = retail.pivot(
    index="customer_id",
    columns="stock_code",
    values="quantity",
).fillna(0)
presence = quantities.gt(0)

example_customers = [15032, 12748, 12839, 12867]
example_products = ["20719", "20724", "20725", "20726", "20727", "20971", "21231", "21232"]
quantity_example = (
    quantities.loc[example_customers, example_products]
    .astype(int)
    .rename_axis(index="Customer", columns="Product")
)
quantity_example

`pivot` creates one row per customer and one column per product. `.fillna(0)` records that a customer did not buy a product when that customer-product pair is absent. `.loc[...]` selects the rows and columns shown here, while `.astype(int)` displays the quantities without decimal places. For example, customer `15032` bought three units each of products `20725`, `20726`, and `20727`, but none of product `20719`.

The presence representation keeps the same rows and columns but reduces each quantity to whether a purchase occurred.

In [ ]:
presence_example = (
    presence.loc[example_customers, example_products]
    .astype(int)
    .rename_axis(index="Customer", columns="Product")
)
presence_example

`.gt(0)` created the Boolean matrix by testing whether each quantity was positive. The displayed slice converts `True` and `False` to 1 and 0, so 1 means purchased and 0 means not purchased. Quantity differences disappear, but the pattern of product presence remains.

The complete representations contain every retained customer and product:

In [ ]:
matrix_size = pd.Series(
    {
        "Customer rows": quantities.shape[0],
        "Product columns": quantities.shape[1],
        "Customers in the reference set": quantities.shape[0] - 1,
    },
    name="Value",
)
matrix_size.to_frame()

The same logic now applies to 815 customer rows and 2,375 aligned product columns.

These representations carry the earlier distinction into the full dataset. The quantity matrix supports the cosine question about purchasing proportions, while the Boolean matrix supports the Jaccard question about shared products.

## Two Customer Neighborhoods

SciPy's [`cdist`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html) compares every row in its first matrix with every row in its second. Passing the row for customer `15032` and all 815 customer rows produces 815 comparisons.

`cdist` returns dissimilarities, so lower values mean nearer. We convert each value to similarity with $1-d$, where higher means nearer, then remove customer `15032`'s comparison with itself and rank the other customers.

In [ ]:
def rank_neighbors(matrix, customer_id, metric):
    """Rank the other customers by similarity to one customer of interest."""

    # Compare the selected customer with every row in the same representation.
    selected_row = matrix.loc[[customer_id]].to_numpy()
    comparison_rows = matrix.to_numpy()
    dissimilarities = cdist(selected_row, comparison_rows, metric=metric)[0]

    # Attach each score to its customer identifier and orient it as similarity.
    ranked = pd.DataFrame(
        {
            "customer_id": matrix.index.to_numpy(),
            "similarity": 1 - dissimilarities,
        }
    )

    # Remove the customer's comparison with itself, then rank high similarity first.
    ranked = ranked.loc[ranked["customer_id"].ne(customer_id)]
    return ranked.sort_values(
        ["similarity", "customer_id"],
        ascending=[False, True],
        kind="stable",
    ).reset_index(drop=True)


cosine_ranking = rank_neighbors(quantities, CUSTOMER_OF_INTEREST, metric="cosine")
jaccard_ranking = rank_neighbors(
    presence.astype(bool),
    CUSTOMER_OF_INTEREST,
    metric="jaccard",
)

top_four = pd.DataFrame(
    {
        "Rank": np.arange(1, 5),
        "Cosine customer": cosine_ranking.head(4)["customer_id"].to_numpy(),
        "Cosine similarity": cosine_ranking.head(4)["similarity"].to_numpy(),
        "Jaccard customer": jaccard_ranking.head(4)["customer_id"].to_numpy(),
        "Jaccard similarity": jaccard_ranking.head(4)["similarity"].to_numpy(),
    }
)
top_four.round({"Cosine similarity": 3, "Jaccard similarity": 3})

The function selects one customer row, compares it with every row through `cdist`, and changes each dissimilarity $d$ into similarity $1-d$. It removes the customer's comparison with itself, sorts higher similarities first, and returns the ranked customers. Passing the quantity matrix with `metric="cosine"` and the Boolean presence matrix with `metric="jaccard"` makes the two analytical choices explicit.

The table places two separate rankings side by side. Read the cosine customer and cosine similarity columns together from top to bottom, then read the two Jaccard columns in the same way. Rank 1 is the customer with the highest similarity under that method. Customers appearing on the same row across the two methods share a rank position, but the table is not comparing those two customers with each other. The similarity scores should also be interpreted within their own methods rather than compared directly across cosine and Jaccard.

Under cosine, customer `13534` ranks first with similarity 0.842, followed by `16456` and `13033`. Under Jaccard, customer `16456` ranks first with similarity 0.419, followed by `14282` and `13178`. Customer `16456` is the only customer in both top-three neighborhoods, moving from second under cosine to first under Jaccard. Customer `13534` provides an especially useful contrast: it leads the cosine ranking but falls to fourth under Jaccard. The two definitions of similarity therefore produce meaningfully different neighborhoods from the same purchase records.

The fourth-ranked customer is shown to provide context for the neighborhood boundary. It is not included when we keep only three neighbors.

Let $k=3$, meaning that we keep the first three customers. Three neighbors provide some local context while remaining easy to inspect. Another purpose could justify another value of $k$; three is not universally correct.

### Pause: what does $k=3$ guarantee?

Does membership in the top three guarantee that every returned customer is close or useful? State exactly what the ranking establishes.

##### Answer

The procedure guarantees that these are the three highest-ranked customers in this reference set under the chosen representation and similarity measure. It does not guarantee an external standard of closeness or usefulness. We still need to inspect the cutoff and the original purchase records.

## The Neighborhood Cutoff

Choosing $k=3$ places a boundary between ranks 3 and 4. The ranking table tells us which customers occupy those positions. The next figure focuses on their scores so we can see whether the third customer is clearly separated from the first customer left out.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for axis, ranking, title, color in [
    (axes[0], cosine_ranking, "Cosine on product quantities", "#4477AA"),
    (axes[1], jaccard_ranking, "Jaccard on product presence", "#228833"),
]:
    boundary = ranking.iloc[2:4].copy()
    boundary["rank"] = [3, 4]
    boundary = boundary.iloc[::-1]
    labels = [
        f"Rank {rank}: {customer}"
        for rank, customer in zip(
            boundary["rank"], boundary["customer_id"], strict=True
        )
    ]
    colors = ["#BBBBBB" if rank == 4 else color for rank in boundary["rank"]]
    bars = axis.barh(labels, boundary["similarity"], color=colors, alpha=0.9)
    axis.bar_label(bars, fmt="%.3f", padding=3)
    axis.set_xlim(0, 1)
    axis.set_xlabel("Similarity")
    axis.set_title(title)
    axis.grid(axis="x", alpha=0.2)

fig.suptitle("The boundary created by keeping three neighbors", y=1.03)
fig.tight_layout()
plt.show()

`.iloc[2:4]` selects ranks 3 and 4 because DataFrame positions begin at zero. The colored bar is retained in the neighborhood; the gray bar is the first customer left out. Both axes run from 0 to 1, but each score still has meaning only under the method that produced it.

### Pause: inspect the cutoff

How strongly does rank 3 separate from rank 4 under each method? Does either panel establish that three customers form a natural group?

##### Answer

Cosine places ranks 3 and 4 at 0.536 and 0.530, a difference of only 0.006. Jaccard places them at 0.364 and 0.300, a difference of 0.064. The Jaccard boundary is more clearly separated in these records, but neither panel proves that three is the uniquely correct neighborhood size.

## Purchases Behind the Scores

A ranking tells us the order produced by a method, but it does not explain why a customer received that position. The following table returns to the purchase records for customer `15032` and every customer appearing in either top-three neighborhood. It shows each customer's rank under both methods, total units, number of products bought, and number of products shared with customer `15032`.

In [ ]:
cosine_top_ids = cosine_ranking.head(K)["customer_id"].tolist()
jaccard_top_ids = jaccard_ranking.head(K)["customer_id"].tolist()
neighbor_ids = sorted(set(cosine_top_ids) | set(jaccard_top_ids))
comparison_ids = [CUSTOMER_OF_INTEREST, *neighbor_ids]
interest_presence = presence.loc[CUSTOMER_OF_INTEREST]

customer_details = (
    retail.loc[retail["customer_id"].isin(comparison_ids)]
    .groupby("customer_id")
    .agg(total_units=("quantity", "sum"), products_bought=("stock_code", "nunique"))
    .rename(
        columns={
            "total_units": "Total units",
            "products_bought": "Products bought",
        }
    )
)
customer_details["Products shared with 15032"] = (
    presence.loc[customer_details.index] & interest_presence
).sum(axis=1)

cosine_positions = pd.Series(
    np.arange(1, len(cosine_ranking) + 1),
    index=cosine_ranking["customer_id"].to_numpy(),
)
jaccard_positions = pd.Series(
    np.arange(1, len(jaccard_ranking) + 1),
    index=jaccard_ranking["customer_id"].to_numpy(),
)
customer_details["Cosine rank"] = (
    customer_details.index.to_series().map(cosine_positions).astype("Int64")
)
customer_details["Jaccard rank"] = (
    customer_details.index.to_series().map(jaccard_positions).astype("Int64")
)

customer_details = customer_details.loc[
    comparison_ids,
    [
        "Cosine rank",
        "Jaccard rank",
        "Total units",
        "Products bought",
        "Products shared with 15032",
    ],
].rename_axis("Customer")
customer_details

`groupby` and `agg` summarize the original purchase rows. The `&` operation counts products present for both customers. The two position Series translate each complete ranking into one-based ranks. Customer `15032` has no rank because it was removed from its own neighbor search.

Customer `15032` supplies the baseline: 1,658 units across 28 products. Customer `13534` bought only 136 units, but cosine ranks it first because cosine discounts total scale and responds to the allocation of those units. It shares 9 products with `15032`, placing it fourth under Jaccard. Customer `16456` ranks second under cosine and first under Jaccard; it shares 13 of its 16 products with `15032`, so both definitions find meaningful resemblance.

The remaining customers show how the neighborhoods diverge. Customer `13033` is third under cosine but eleventh under Jaccard, with only 5 shared products. Customer `14282` is seventh under cosine but second under Jaccard, sharing 12 of its 13 products with `15032`. Customer `13178` shares 16 products, the largest count in the table, but also buys 16 products that `15032` does not; Jaccard ranks it third because the union contains all 44 products bought by either customer.

## Quantity Patterns Behind Cosine

The summary table does not show how quantities are distributed across products. To see the pattern cosine uses, the next figure compares customer `15032`, cosine leader `13534`, and Jaccard neighbor `14282` across the eight largest product quantities for `15032`. Each customer's complete quantity row is divided by its Euclidean length before these product positions are displayed. This removes overall scale in the same way used by cosine similarity.

In [ ]:
interest_top_codes = (
    quantities.loc[CUSTOMER_OF_INTEREST]
    .nlargest(8)
    .index
)
code_labels = (
    retail.drop_duplicates("stock_code")
    .set_index("stock_code")
    .loc[interest_top_codes, "description"]
    .str.replace(r"\s+", " ", regex=True)
)
short_labels = [label.replace("JUMBO ", "").title() for label in code_labels]

profile_ids = [CUSTOMER_OF_INTEREST, 13534, 14282]
profile_lengths = np.linalg.norm(quantities.loc[profile_ids], axis=1)
normalized_profiles = quantities.loc[profile_ids].div(profile_lengths, axis=0)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.8), sharex=True, sharey=True)

for axis, customer_id in zip(axes, profile_ids, strict=True):
    values = normalized_profiles.loc[customer_id, interest_top_codes]
    axis.barh(short_labels, values, color=PRODUCT_COLORS * 2, edgecolor="black", linewidth=0.4)
    axis.set_title(f"Customer {customer_id}")
    axis.set_xlabel("Quantity divided by profile length")
    axis.grid(axis="x", alpha=0.2)

axes[0].invert_yaxis()
fig.suptitle("Normalized quantities for customer 15032's eight largest purchases", y=1.02)
fig.tight_layout()
plt.show()

`.nlargest(8)` selects customer `15032`'s eight largest product quantities. `np.linalg.norm` calculates each complete profile's Euclidean length, and `.div(..., axis=0)` divides every quantity in that row by its length. A missing bar means that the customer did not buy that product. The ranking still uses all 2,375 product columns; the figure shows the eight positions that contribute most strongly to customer `15032`'s profile.

### Pause: interpret the quantity patterns

Which comparison customer allocates more of its normalized quantity to the same prominent products as customer `15032`? Which customer buys several of those products but distributes quantity differently? What cosine ordering do you expect?

##### Answer

Customer `13534` places its largest normalized quantities in several of the same product positions as customer `15032`, so we expect it to rank ahead of `14282` under cosine. Customer `14282` buys several of the displayed products but allocates its quantities differently.

### Cosine Interpretation

Customer `13534` has far lower total volume than customer `15032`, but their normalized quantity patterns align closely enough to produce cosine similarity 0.842 and rank 1. Customer `14282` shares many products with `15032`, but its different quantity allocation places it at cosine rank 7. This figure explains the main quantity pattern behind that contrast without pretending that eight displayed products are the complete calculation.

## Product Overlap Behind Jaccard

Jaccard requires a different view because it ignores purchase quantities. For each of the first four Jaccard customers, the next figure divides the union of purchased products into three parts: products shared with customer `15032`, products bought only by `15032`, and products bought only by the comparison customer. The shared part as a fraction of the complete bar is the Jaccard similarity.

In [ ]:
jaccard_cases = jaccard_ranking.head(4)["customer_id"].tolist()
overlap_rows = []

for customer_id in jaccard_cases:
    customer_presence = presence.loc[customer_id]
    shared = int((interest_presence & customer_presence).sum())
    only_interest = int((interest_presence & ~customer_presence).sum())
    only_customer = int((~interest_presence & customer_presence).sum())
    overlap_rows.append(
        {
            "Customer": customer_id,
            "Shared": shared,
            "Only 15032": only_interest,
            "Only comparison customer": only_customer,
        }
    )

overlap = pd.DataFrame(overlap_rows).set_index("Customer")
labels = [f"Rank {rank}: {customer}" for rank, customer in enumerate(overlap.index, start=1)]

fig, axis = plt.subplots(figsize=(9, 4.8))
left = np.zeros(len(overlap))
for column, color in [
    ("Shared", "#228833"),
    ("Only 15032", "#BBBBBB"),
    ("Only comparison customer", "#CC6677"),
]:
    axis.barh(labels, overlap[column], left=left, label=column, color=color)
    left += overlap[column].to_numpy()

axis.invert_yaxis()
axis.set_xlabel("Products in the union")
axis.set_title("Product overlap with customer 15032")
axis.legend(frameon=False)
axis.grid(axis="x", alpha=0.2)
fig.tight_layout()
plt.show()

The `&` operator identifies shared purchases. The `~` operator reverses `True` and `False`, allowing the code to count products bought by only one customer. Stacking the three counts makes each full bar the union used in the Jaccard denominator.

### Pause: interpret product overlap

Which customers devote the largest fractions of their union bars to shared products? Why can customer `13178` share more products than customer `16456` but receive a lower Jaccard similarity?

##### Answer

Customers `16456` and `14282` have the largest shared fractions. Customer `13178` shares 16 products but also buys 16 products that customer `15032` does not, enlarging the union and reducing the shared fraction.

### Jaccard Interpretation

Customer `16456` shares 13 of the 31 products in its union with customer `15032`, producing Jaccard similarity 0.419. Customer `14282` shares 12 of 29, producing 0.414. Customer `13178` shares 16 of 44, producing 0.364. Customer `13534` shares 9 of 30, producing 0.300 and rank 4. Jaccard rewards the proportion of the union that is shared, not the shared count alone.

## Comparing the Neighborhoods

We can now return to the manager's request. Cosine returns customers `13534`, `16456`, and `13033` because their quantity profiles point in the most similar directions. Jaccard returns customers `16456`, `14282`, and `13178` because they have the largest shared fractions of purchased products. Customer `16456` is the only customer retained in both top-three neighborhoods.

The disagreement is not an error to resolve by choosing the list with larger numerical scores. The two methods were asked different questions. The manager's phrase “bought most like” does not tell us whether quantity allocation or product overlap matters more for the intended use.

### Pause: answer the manager

What should we tell the manager instead of presenting one neighborhood as the uniquely correct answer? Include what each list preserves and what decision is still needed.

##### Answer

We should provide both labeled lists and explain that cosine preserves the allocation of quantities while Jaccard preserves product overlap. Customer `16456` is supported by both definitions. Before using one complete list, the manager must decide which kind of resemblance serves the customer review.

## The Main Result

The manager asked for three customers who bought most like customer `15032`, but the request does not determine one list until *bought most like* is defined.

- If the purpose requires a similar allocation of quantities across products, cosine identifies customers `13534`, `16456`, and `13033`.
- If the purpose requires purchases from a similar set of products, Jaccard identifies customers `16456`, `14282`, and `13178`.
- Customer `16456` appears in both lists, while the other four customers show how much the answer depends on the chosen meaning of similarity.

These are the three highest-ranked customers under each method in this December UK reference set. The rankings do not establish that every returned customer is close by an external standard, that (k=3) is the only reasonable cutoff, that either neighborhood will persist in another period, or that one definition fits every business purpose.

A neighborhood is not a fixed property waiting to be found in the data. It is the result of choices about what one row represents, which information the representation keeps, how resemblance is measured, which observations are available for comparison, and how many neighbors are retained. The scores and original records help us explain the result and decide whether the returned observations are useful for the purpose at hand.

KNN regression and classification use the same neighborhood idea as an intermediate step toward a prediction. Here, the neighborhood itself is the result we inspect and explain.